In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from scipy.stats import norm


# ============================================================
# 1) AUTO-GENERATE CSV FILES
# ============================================================

def generate_supply_chain_csvs(
    out_dir="data",
    n_days=120,
    skus=("SKU_A", "SKU_B", "SKU_C"),
    start_date="2024-01-01",
    seed=42,
):
    rng = np.random.default_rng(seed)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    dates = pd.date_range(start=start_date, periods=n_days, freq="D")

    sales_rows = []
    inventory_rows = []
    params_rows = []

    for sku in skus:
        base = rng.integers(8, 25)
        trend = rng.uniform(-0.03, 0.08)
        weekly_amp = rng.uniform(0.1, 0.3)
        yearly_amp = rng.uniform(0.05, 0.15)
        noise_std = rng.uniform(1.5, 3.5)

        sku_demands = []
        for i, d in enumerate(dates):
            weekly = 1.0 + weekly_amp * np.sin(2 * np.pi * d.dayofweek / 7)
            yearly = 1.0 + yearly_amp * np.sin(2 * np.pi * d.dayofyear / 365)
            level = base * weekly * yearly + trend * i

            promo = rng.integers(10, 25) if rng.random() < 0.06 else 0
            noise = rng.normal(0, noise_std)

            qty = max(0, int(round(level + promo + noise)))
            sku_demands.append(qty)
            sales_rows.append([d, sku, qty])

        avg_demand = max(1, np.mean(sku_demands))
        opening_stock = int(round(avg_demand * rng.uniform(7, 12)))
        inventory_rows.append([sku, opening_stock])

        unit_cost = round(rng.uniform(5, 20), 2)
        holding_cost = round(rng.uniform(0.03, 0.12), 3)
        stockout_cost = round(rng.uniform(2, 6), 2)
        lead_time_days = int(rng.integers(3, 10))
        min_order_qty = int(rng.integers(20, 60))
        service_level = round(rng.uniform(0.90, 0.98), 2)

        params_rows.append([
            sku,
            unit_cost,
            holding_cost,
            stockout_cost,
            lead_time_days,
            min_order_qty,
            service_level,
        ])

    sales = pd.DataFrame(sales_rows, columns=["date", "sku", "qty_sold"])
    inventory = pd.DataFrame(inventory_rows, columns=["sku", "opening_stock"])
    params = pd.DataFrame(
        params_rows,
        columns=[
            "sku",
            "unit_cost",
            "holding_cost_per_day",
            "stockout_cost",
            "lead_time_days",
            "min_order_qty",
            "service_level",
        ],
    )

    sales.to_csv(out_dir / "sales.csv", index=False)
    inventory.to_csv(out_dir / "inventory.csv", index=False)
    params.to_csv(out_dir / "params.csv", index=False)

    print(f"Generated files in: {out_dir.resolve()}")
    return sales, inventory, params


# ============================================================
# 2) LOAD DATA
# ============================================================

def load_data(data_dir="data"):
    data_dir = Path(data_dir)
    sales = pd.read_csv(data_dir / "sales.csv", parse_dates=["date"])
    inventory = pd.read_csv(data_dir / "inventory.csv")
    params = pd.read_csv(data_dir / "params.csv")
    return sales, inventory, params


# ============================================================
# 3) PREP DAILY DEMAND
# ============================================================

def prepare_daily_demand(sales: pd.DataFrame) -> pd.DataFrame:
    daily = (
        sales.groupby(["date", "sku"], as_index=False)["qty_sold"]
        .sum()
        .sort_values(["sku", "date"])
        .reset_index(drop=True)
    )
    return daily


# ============================================================
# 4) SIMPLE EWMA FORECAST PER SKU
#    Forecast for day t uses demand history up to t-1
# ============================================================

def add_ewma_forecast(daily: pd.DataFrame, alpha=0.3) -> pd.DataFrame:
    out = daily.copy()
    out["forecast"] = np.nan
    out["forecast_error"] = np.nan

    for sku, g in out.groupby("sku"):
        idx = g.index
        actuals = g["qty_sold"].values

        forecasts = np.zeros(len(actuals), dtype=float)
        forecasts[0] = actuals[0]

        for i in range(1, len(actuals)):
            forecasts[i] = alpha * actuals[i - 1] + (1 - alpha) * forecasts[i - 1]

        out.loc[idx, "forecast"] = forecasts
        out.loc[idx, "forecast_error"] = g["qty_sold"].values - forecasts

    return out


# ============================================================
# 5) POLICY HELPERS
# ============================================================

@dataclass
class SkuPolicy:
    sku: str
    opening_stock: int
    holding_cost_per_day: float
    stockout_cost: float
    lead_time_days: int
    min_order_qty: int
    service_level: float
    order_cost_fixed: float = 0.0
    review_period_days: int = 1


def build_sku_policies(inventory: pd.DataFrame, params: pd.DataFrame) -> dict:
    merged = inventory.merge(params, on="sku", how="inner")
    policies = {}

    for _, row in merged.iterrows():
        policies[row["sku"]] = SkuPolicy(
            sku=row["sku"],
            opening_stock=int(row["opening_stock"]),
            holding_cost_per_day=float(row["holding_cost_per_day"]),
            stockout_cost=float(row["stockout_cost"]),
            lead_time_days=int(row["lead_time_days"]),
            min_order_qty=int(row["min_order_qty"]),
            service_level=float(row["service_level"]),
            order_cost_fixed=0.0,
            review_period_days=1,
        )

    return policies


def rolling_forecast_sigma(errors: pd.Series, window=14, fallback=1.0) -> pd.Series:
    sigma = errors.shift(1).rolling(window=window, min_periods=3).std()
    sigma = sigma.fillna(fallback)
    sigma = sigma.clip(lower=0.1)
    return sigma


# ============================================================
# 6) INVENTORY SIMULATION
# ============================================================

class InventorySystem:
    def __init__(self, policy: SkuPolicy):
        self.policy = policy
        self.on_hand = float(policy.opening_stock)
        self.backlog = 0.0

        self.pipeline = {}  # arrival_date -> qty
        self.total_holding_cost = 0.0
        self.total_stockout_cost = 0.0
        self.total_order_cost = 0.0

        self.total_demand = 0.0
        self.total_filled = 0.0

        self.log = []

    def inventory_position(self):
        on_order = sum(self.pipeline.values())
        return self.on_hand + on_order - self.backlog

    def receive_orders(self, date):
        arriving = self.pipeline.pop(date, 0.0)
        self.on_hand += arriving
        return arriving

    def place_order(self, date, qty):
        if qty <= 0:
            return None

        arrival_date = date + pd.Timedelta(days=self.policy.lead_time_days)
        self.pipeline[arrival_date] = self.pipeline.get(arrival_date, 0.0) + qty
        self.total_order_cost += self.policy.order_cost_fixed
        return arrival_date

    def satisfy_demand(self, demand):
        total_required = self.backlog + demand
        shipped = min(self.on_hand, total_required)

        self.on_hand -= shipped
        remaining = total_required - shipped
        self.backlog = remaining

        filled_today = min(shipped, demand)
        self.total_demand += demand
        self.total_filled += filled_today

        return shipped, self.backlog

    def apply_costs(self):
        self.total_holding_cost += self.policy.holding_cost_per_day * max(self.on_hand, 0)
        self.total_stockout_cost += self.policy.stockout_cost * self.backlog

    def fill_rate(self):
        if self.total_demand == 0:
            return 1.0
        return self.total_filled / self.total_demand

    def total_cost(self):
        return self.total_holding_cost + self.total_stockout_cost + self.total_order_cost


# ============================================================
# 7) AGENT DECISION LOGIC
#    Project over lead time + review period
# ============================================================

def decide_order(
    current_date: pd.Timestamp,
    system: InventorySystem,
    policy: SkuPolicy,
    today_forecast: float,
    today_sigma: float,
):
    protection_days = policy.lead_time_days + policy.review_period_days
    z = norm.ppf(policy.service_level)

    expected_demand_protection = today_forecast * protection_days
    safety_stock = z * today_sigma * np.sqrt(protection_days)

    reorder_point = expected_demand_protection + safety_stock
    target_level = reorder_point + today_forecast * policy.review_period_days

    inv_pos = system.inventory_position()

    if inv_pos <= reorder_point:
        raw_qty = target_level - inv_pos
        order_qty = max(policy.min_order_qty, int(np.ceil(raw_qty)))
        action = "ORDER"
        rationale = (
            f"InvPos {inv_pos:.1f} <= ROP {reorder_point:.1f}; "
            f"forecast={today_forecast:.1f}, sigma={today_sigma:.1f}, "
            f"safety_stock={safety_stock:.1f}, target={target_level:.1f}"
        )
    else:
        order_qty = 0
        action = "WAIT"
        rationale = (
            f"InvPos {inv_pos:.1f} > ROP {reorder_point:.1f}; "
            f"enough stock through protection window"
        )

    return {
        "action": action,
        "qty": order_qty,
        "inventory_position": inv_pos,
        "forecast": today_forecast,
        "sigma": today_sigma,
        "protection_days": protection_days,
        "safety_stock": safety_stock,
        "reorder_point": reorder_point,
        "target_level": target_level,
        "rationale": rationale,
    }


# ============================================================
# 8) RUN SIMULATION FOR ALL SKUS
# ============================================================

def simulate_inventory_agent(daily_fcst: pd.DataFrame, policies: dict, sigma_window=14):
    results = []
    daily_logs = []

    for sku, g in daily_fcst.groupby("sku"):
        g = g.sort_values("date").copy().reset_index(drop=True)
        policy = policies[sku]
        system = InventorySystem(policy)

        fallback_sigma = max(1.0, g["forecast_error"].std(ddof=1) if len(g) > 1 else 1.0)
        g["sigma_hat"] = rolling_forecast_sigma(
            g["forecast_error"],
            window=sigma_window,
            fallback=fallback_sigma,
        )

        for _, row in g.iterrows():
            date = row["date"]
            demand = float(row["qty_sold"])
            forecast = max(0.0, float(row["forecast"]))
            sigma = max(0.1, float(row["sigma_hat"]))

            arrivals = system.receive_orders(date)

            decision = decide_order(
                current_date=date,
                system=system,
                policy=policy,
                today_forecast=forecast,
                today_sigma=sigma,
            )

            arrival_date = None
            if decision["qty"] > 0:
                arrival_date = system.place_order(date, decision["qty"])

            shipped, backlog = system.satisfy_demand(demand)
            system.apply_costs()

            log_row = {
                "date": date,
                "sku": sku,
                "demand": demand,
                "forecast": forecast,
                "sigma_hat": sigma,
                "arrivals": arrivals,
                "on_hand_end": system.on_hand,
                "backlog_end": backlog,
                "inventory_position_pre_demand": decision["inventory_position"],
                "reorder_point": decision["reorder_point"],
                "target_level": decision["target_level"],
                "action": decision["action"],
                "order_qty": decision["qty"],
                "po_arrival_date": arrival_date,
                "rationale": decision["rationale"],
            }
            daily_logs.append(log_row)

        results.append({
            "sku": sku,
            "fill_rate": system.fill_rate(),
            "holding_cost": system.total_holding_cost,
            "stockout_cost": system.total_stockout_cost,
            "order_cost": system.total_order_cost,
            "total_cost": system.total_cost(),
            "ending_on_hand": system.on_hand,
            "ending_backlog": system.backlog,
            "total_demand": system.total_demand,
            "total_filled": system.total_filled,
        })

    results_df = pd.DataFrame(results).sort_values("sku").reset_index(drop=True)
    logs_df = pd.DataFrame(daily_logs).sort_values(["sku", "date"]).reset_index(drop=True)
    return results_df, logs_df


# ============================================================
# 9) MAIN SCRIPT
# ============================================================

def main():
    # Step A: generate CSVs
    sales, inventory, params = generate_supply_chain_csvs(
        out_dir="data",
        n_days=90,
        skus=("SKU_A", "SKU_B", "SKU_C"),
        start_date="2024-01-01",
        seed=7,
    )

    # Step B: load CSVs (already done by generate_supply_chain_csvs returning them)
    # sales, inventory, params = load_data("data")

    # Step C: aggregate demand
    daily = prepare_daily_demand(sales)

    # Step D: add EWMA forecast
    daily_fcst = add_ewma_forecast(daily, alpha=0.35)

    # Step E: build SKU policies
    policies = build_sku_policies(inventory, params)

    # Step F: simulate
    results_df, logs_df = simulate_inventory_agent(daily_fcst, policies, sigma_window=14)

    # Step G: print summary
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)

    print("\n=== inventory.csv ===")
    print(inventory)

    print("\n=== params.csv ===")
    print(params)

    print("\n=== Daily Agent Log (first 30 rows) ===")
    print(
        logs_df[
            [
                "date",
                "sku",
                "demand",
                "forecast",
                "sigma_hat",
                "arrivals",
                "on_hand_end",
                "backlog_end",
                "inventory_position_pre_demand",
                "reorder_point",
                "action",
                "order_qty",
                "po_arrival_date",
                "rationale",
            ]
        ].head(30).to_string(index=False)
    )

    print("\n=== KPI Summary by SKU ===")
    print(results_df.to_string(index=False))

    print("\n=== Portfolio Totals ===")
    print(
        results_df[["holding_cost", "stockout_cost", "order_cost", "total_cost"]]
        .sum()
        .to_frame("value")
    )

    # Optional: save outputs (handled in the if __name__ == "__main__" block now)
    # daily_fcst.to_csv("data/daily_with_forecast.csv", index=False)
    # logs_df.to_csv("data/agent_daily_log.csv", index=False)
    # results_df.to_csv("data/agent_kpis.csv", index=False)

    # print("\nSaved:")
    # print("- data/daily_with_forecast.csv")
    # print("- data/agent_daily_log.csv")
    # print("- data/agent_kpis.csv")
    return daily, inventory, params, results_df, logs_df



# =========================
# BASELINE POLICY COMPARISON
# =========================

# Removed the old InventorySim definition and replaced with adaptation of InventorySystem

def run_baseline_policy(daily_df, inventory_df, params_df):
    """
    Simple baseline:
    - For each SKU, compute average daily demand from history
    - Use static reorder point:
        ROP = mean_demand * lead_time + z * sigma * sqrt(lead_time)
    - Order a fixed quantity when inventory position <= ROP
    - Fixed quantity = max(min_order_qty, round(mean_demand * (lead_time + 3)))
    """
    logs = []
    kpis = []

    for sku_name in inventory_df["sku"]:
        sku_sales = daily_df[daily_df["sku"] == sku_name].sort_values("date").reset_index(drop=True)
        inv_row = inventory_df[inventory_df["sku"] == sku_name].iloc[0]
        p_row = params_df[params_df["sku"] == sku_name].iloc[0]

        # Create a SkuPolicy for the baseline simulation
        baseline_policy = SkuPolicy(
            sku=sku_name,
            opening_stock=int(inv_row["opening_stock"]),
            holding_cost_per_day=float(p_row["holding_cost_per_day"]),
            stockout_cost=float(p_row["stockout_cost"]),
            lead_time_days=int(p_row["lead_time_days"]),
            min_order_qty=int(p_row["min_order_qty"]),
            service_level=float(p_row["service_level"]),
            order_cost_fixed=0.0, # Assuming 0 for fixed order cost in baseline for simplicity
            review_period_days=1 # Baseline often assumes daily review
        )

        system = InventorySystem(baseline_policy)

        lead_time = baseline_policy.lead_time_days
        mean_demand = sku_sales["qty_sold"].mean()
        sigma = sku_sales["qty_sold"].std(ddof=1)
        sigma = 1.0 if pd.isna(sigma) or sigma == 0 else sigma

        z = norm.ppf(baseline_policy.service_level)
        safety_stock = z * sigma * np.sqrt(lead_time)
        reorder_point = mean_demand * lead_time + safety_stock

        fixed_order_qty = max(baseline_policy.min_order_qty, int(round(mean_demand * (lead_time + 3))))

        # Reset totals for each SKU's simulation
        total_demand = 0.0
        total_filled = 0.0

        for _, row in sku_sales.iterrows():
            date = row["date"]
            demand = float(row["qty_sold"])

            arrivals = system.receive_orders(date) # Process arrivals first

            inventory_position_pre_demand = system.inventory_position()

            order_qty = 0
            action = "WAIT"
            po_arrival_date = pd.NaT
            rationale = ""

            if inventory_position_pre_demand <= reorder_point:
                order_qty = fixed_order_qty
                action = "ORDER"
                po_arrival_date = system.place_order(date, order_qty) # Place order with the system
                rationale = (
                    f"InvPos {inventory_position_pre_demand:.1f} <= static ROP {reorder_point:.1f}; "
                    f"place fixed order qty {order_qty}"
                )
            else:
                rationale = (
                    f"InvPos {inventory_position_pre_demand:.1f} > static ROP {reorder_point:.1f}; "
                    f"wait"
                )

            shipped, backlog_end = system.satisfy_demand(demand)
            system.apply_costs()

            total_demand += demand
            total_filled += shipped # shipped from satisfy_demand is what was actually filled for current day's demand + backlog

            logs.append({
                "date": date,
                "sku": sku_name,
                "policy": "baseline",
                "demand": demand,
                "forecast": mean_demand,
                "sigma_hat": sigma,
                "arrivals": arrivals,
                "on_hand_end": system.on_hand,
                "backlog_end": backlog_end,
                "inventory_position_pre_demand": inventory_position_pre_demand,
                "reorder_point": reorder_point,
                "action": action,
                "order_qty": order_qty,
                "po_arrival_date": po_arrival_date,
                "rationale": rationale
            })

        fill_rate = system.fill_rate() # Use system's fill_rate

        kpis.append({
            "sku": sku_name,
            "policy": "baseline",
            "fill_rate": fill_rate,
            "holding_cost": system.total_holding_cost,
            "stockout_cost": system.total_stockout_cost,
            "order_cost": system.total_order_cost,
            "total_cost": system.total_cost(),
            "ending_on_hand": system.on_hand,
            "ending_backlog": system.backlog,
            "total_demand": total_demand,
            "total_filled": total_filled
        })

    return pd.DataFrame(logs), pd.DataFrame(kpis)


if __name__ == "__main__":
    # Run the main simulation and capture its outputs
    daily_data, inventory_data, params_data, agent_kpis_df, agent_logs_df = main()

    # Define data_dir here as it's used for saving
    data_dir = Path("data")

    # Save outputs from agent simulation
    # daily_fcst (from main's scope) is not directly returned, but it saves to data/daily_with_forecast.csv
    # If needed, daily_fcst could be reloaded from the CSV, or main could return it.
    # For this fix, we assume main() has done its saving duty for daily_with_forecast.csv
    # Let's reload daily_fcst to ensure it's available for saving just in case main was modified to not save it.
    # However, since main is executing the print and saving logic, it's safer to pass the actual `daily_fcst` back.
    # But since it's already saved by main(), I will just proceed with the comparison.
    # I will add `daily_fcst` as a return from `main` to be consistent and allow saving outside.

    # Rerunning main from scratch and then capturing outputs
    # Re-generate CSVs and run agent simulation directly to make outputs explicitly available.
    sales, inventory, params = generate_supply_chain_csvs(
        out_dir="data",
        n_days=90,
        skus=("SKU_A", "SKU_B", "SKU_C"),
        start_date="2024-01-01",
        seed=7,
    )
    daily = prepare_daily_demand(sales)
    daily_fcst = add_ewma_forecast(daily, alpha=0.35)
    policies = build_sku_policies(inventory, params)
    agent_kpis_df, agent_logs_df = simulate_inventory_agent(daily_fcst, policies, sigma_window=14)

    # Print agent summary (as originally in main)
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)

    print("\n=== inventory.csv ===")
    print(inventory)

    print("\n=== params.csv ===")
    print(params)

    print("\n=== Daily Agent Log (first 30 rows) ===")
    print(
        agent_logs_df[
            [
                "date",
                "sku",
                "demand",
                "forecast",
                "sigma_hat",
                "arrivals",
                "on_hand_end",
                "backlog_end",
                "inventory_position_pre_demand",
                "reorder_point",
                "action",
                "order_qty",
                "po_arrival_date",
                "rationale",
            ]
        ].head(30).to_string(index=False)
    )

    print("\n=== KPI Summary by SKU ===")
    print(agent_kpis_df.to_string(index=False))

    print("\n=== Portfolio Totals ===")
    print(
        agent_kpis_df[["holding_cost", "stockout_cost", "order_cost", "total_cost"]]
        .sum()
        .to_frame("value")
    )

    # Save outputs for agent
    daily_fcst.to_csv(data_dir / "daily_with_forecast.csv", index=False)
    agent_logs_df.to_csv(data_dir / "agent_daily_log.csv", index=False)
    agent_kpis_df.to_csv(data_dir / "agent_kpis.csv", index=False)

    print("\nSaved:")
    print(f"- {data_dir / 'daily_with_forecast.csv'}")
    print(f"- {data_dir / 'agent_daily_log.csv'}")
    print(f"- {data_dir / 'agent_kpis.csv'}")


    # Assign variables for baseline comparison
    daily_df = daily # The raw aggregated daily demand
    inventory_df = inventory
    params_df = params
    kpis_df = agent_kpis_df # Agent KPIs for comparison

    # Run baseline
    baseline_log_df, baseline_kpis_df = run_baseline_policy(daily_df, inventory_df, params_df)

    print("\n=== Baseline KPI Summary by SKU ===")
    print(baseline_kpis_df.to_string(index=False))

    # Compare baseline vs agent
    agent_compare = kpis_df.copy()
    agent_compare["policy"] = "agent"

    comparison_df = pd.concat(
        [baseline_kpis_df, agent_compare],
        ignore_index=True
    ).sort_values(["sku", "policy"])

    print("\n=== Baseline vs Agent Comparison ===")
    print(comparison_df[[
        "sku", "policy", "fill_rate", "holding_cost",
        "stockout_cost", "order_cost", "total_cost"
    ]].to_string(index=False))


    # Wide comparison table
    comparison_wide = comparison_df.pivot_table(
        index="sku",
        columns="policy",
        values=["fill_rate", "holding_cost", "stockout_cost", "total_cost"],
        aggfunc="first"
    )

    print("\n=== Wide Comparison Table ===")
    print(comparison_wide)


    # Optional: save outputs
    baseline_log_df.to_csv(data_dir / "baseline_daily_log.csv", index=False)
    baseline_kpis_df.to_csv(data_dir / "baseline_kpis.csv", index=False)
    comparison_df.to_csv(data_dir / "baseline_vs_agent_comparison.csv", index=False)

    print("\nSaved:")
    print(f"- {data_dir / 'baseline_daily_log.csv'}")
    print(f"- {data_dir / 'baseline_kpis.csv'}")
    print(f"- {data_dir / 'baseline_vs_agent_comparison.csv'}")


Generated files in: /content/data

=== inventory.csv ===
     sku  opening_stock
0  SKU_A            232
1  SKU_B             71
2  SKU_C            257

=== params.csv ===
     sku  unit_cost  holding_cost_per_day  stockout_cost  lead_time_days  min_order_qty  service_level
0  SKU_A      13.51                 0.034           4.36               4             24           0.95
1  SKU_B       8.55                 0.072           5.52               9             24           0.97
2  SKU_C       8.21                 0.116           5.54               4             25           0.93

=== Daily Agent Log (first 30 rows) ===
      date   sku  demand  forecast  sigma_hat  arrivals  on_hand_end  backlog_end  inventory_position_pre_demand  reorder_point action  order_qty po_arrival_date                                                                             rationale
2024-01-01 SKU_A    24.0 24.000000   8.114193       0.0        208.0          0.0                          232.0     149.84403